In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = next(
    p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "utils" / "paths.py").exists()
)

sys.path.insert(0, str(PROJECT_ROOT))

from utils.paths import MODEL_PATH, TEST_DATA

In [2]:
import pandas as pd
import joblib

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

TEST_PATH = TEST_DATA

# Test data
df = pd.read_csv(TEST_PATH)

# Target
y_test = df["Machine failure"]

# Features
X_test = df.drop(columns=["Machine failure"])

# Load model
model = joblib.load(
   MODEL_PATH
)

# Rename columns exactly like training data
X_test = X_test.rename(columns={
    "Air temperature [K]": "Air_temperature_K",
    "Process temperature [K]": "Process_temperature_K",
    "Rotational speed [rpm]": "Rotational_speed_rpm",
    "Torque [Nm]": "Torque_Nm",
    "Tool wear [min]": "Tool_wear_min"
})

# Keep only columns used during training
X_test = X_test[model.feature_names_in_]

print("Test columns:")
print(X_test.columns.tolist())

# Prediction
y_pred = model.predict(X_test)

y_probability = model.predict_proba(X_test)[:, 1]

# Metrics
accuracy = accuracy_score(y_test, y_pred)

precision = precision_score(
    y_test,
    y_pred,
    zero_division=0
)

recall = recall_score(
    y_test,
    y_pred,
    zero_division=0
)

f1 = f1_score(
    y_test,
    y_pred,
    zero_division=0
)

roc_auc = roc_auc_score(
    y_test,
    y_probability
)

print("\n===== MODEL EVALUATION =====")
print("Accuracy :", accuracy)
print("Precision:", precision)
print("Recall   :", recall)
print("F1 Score :", f1)
print("ROC-AUC  :", roc_auc)

print("\n===== CONFUSION MATRIX =====")
print(confusion_matrix(y_test, y_pred))

print("\n===== CLASSIFICATION REPORT =====")
print(classification_report(
    y_test,
    y_pred,
    zero_division=0
))

Test columns:
['Air_temperature_K', 'Process_temperature_K', 'Rotational_speed_rpm', 'Torque_Nm', 'Tool_wear_min', 'Type_H', 'Type_L', 'Type_M']

===== MODEL EVALUATION =====
Accuracy : 0.978
Precision: 0.6428571428571429
Recall   : 0.7941176470588235
F1 Score : 0.7105263157894737
ROC-AUC  : 0.973122944830106

===== CONFUSION MATRIX =====
[[1902   30]
 [  14   54]]

===== CLASSIFICATION REPORT =====
              precision    recall  f1-score   support

           0       0.99      0.98      0.99      1932
           1       0.64      0.79      0.71        68

    accuracy                           0.98      2000
   macro avg       0.82      0.89      0.85      2000
weighted avg       0.98      0.98      0.98      2000

